# Wstęp do uczenia maszynowego - Notebook 3, wersja dla studentów
**Autor: Michał Ciach**
Zadania oznaczone gwiazdką \* są opcjonalne. Mogą być trudniejsze lub bardziej czasochłonne.


## Opis
Na dzisiejszych zajęciach poznamy metodę wnioskowania statystycznego zwaną *testowaniem hipotez*.

*Czas na teorię.* Ogólna idea testowania hipotez statystycznych polega na postawieniu hipotezy statystycznej - np. że pewien parametr jest większy od 0 - i próbie *jej odrzucenia* poprzez pokazanie, że zaobserwowanie naszych danych byłoby bardzo mało prawdopodobne, gdyby ta hipoteza była prawdziwa.

Dlaczego odrzucamy hipotezy statystyczne, zamiast je potwierdzać? Ponieważ w ogólności potwierdzenie ich na podstawie prób losowych jest niemożliwe. Załóżmy na przykład, że badamy toksyczność nowego leku, podając go 10 szczurom i obserwując je przez tydzień. To, że szczury nie zachorowały, może sugerować, że lek jest bezpieczny, ale tego nie dowodzi - w końcu to tylko 10 zwierząt obserwowanych przez krótki czas. Z drugiej strony, jeśli wszystkie 10 zwierząt zachoruje (a grupa kontrolna będzie zdrowa), możemy zdecydowanie odrzucić hipotezę, że lek jest bezpieczny.

Inny przykład: załóżmy, że mamy hipotezę, że średnia log-długość ludzkiego białka wynosi 2.30, i losujemy 200 białek, aby ją zweryfikować. Jeśli ich średnia log-długość okaże się zbliżona do 2.30 - na przykład 2.31 z odchyleniem standardowym 0.1 - nie dowodzi to naszej hipotezy. Prawdziwa średnia log-długość może nadal wynosić na przykład 2.29 albo 2.305. Jeśli jednak średnia log-długość w naszej próbie wyniesie 2.70 z odchyleniem standardowym 0.1, jest to mocny dowód na to, że nasza hipoteza jest fałszywa.

Ogólniej: to, że dane w pewnym stopniu zgadzają się z naszymi przekonaniami, nie dowodzi, że nasze przekonania są prawdziwe. Jeśli jednak dane przeczą naszym przekonaniom, to dowodzi to, że nasze przekonania są fałszywe.






## Dane i import bibliotek

In [1]:
!pip install gdown
!pip install --upgrade gdown

In [2]:
!gdown https://drive.google.com/uc?id=1lSiX67iYeqYO8S5251ekN3y7TJ84RXUy
!gdown https://drive.google.com/uc?id=1UmqDDwBWfhQt5gIbp1rfw69m4FMIGx7w

Downloading...
From: https://drive.google.com/uc?id=1lSiX67iYeqYO8S5251ekN3y7TJ84RXUy
To: /content/3. citizen incomes.tsv
100% 40.6k/40.6k [00:00<00:00, 34.2MB/s]
Downloading...
From: https://drive.google.com/uc?id=1UmqDDwBWfhQt5gIbp1rfw69m4FMIGx7w
To: /content/3. protein_lengths.tsv
100% 29.3M/29.3M [00:00<00:00, 101MB/s]


In [3]:
!pip install --upgrade scipy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.4/38.4 MB 17.5 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.11.4
    Uninstalling scipy-1.11.4:
      Successfully uninstalled scipy-1.11.4


In [4]:
import pandas as pd
import numpy as np
import plotly.express as px
from scipy.stats import t as tstud
from scipy.stats import ttest_ind, ttest_rel, ttest_1samp, norm, kstest, mannwhitneyu, shapiro, chisquare, chi2_contingency
import plotly.graph_objects as go
from statsmodels.stats.multitest import fdrcorrection

In [5]:
protein_lengths = pd.read_csv('3. protein_lengths.tsv', sep='\t')
protein_lengths['LogLength'] = np.log10(protein_lengths['Protein length'])
protein_lengths

,Scientific name,Common name,Protein ID,Protein length,LogLength
0,Homo sapiens,Human,NP_000005.3,1474,3.168497
1,Homo sapiens,Human,NP_000006.2,290,2.462398
2,Homo sapiens,Human,NP_000007.1,421,2.624282
3,Homo sapiens,Human,NP_000008.1,412,2.614897
4,Homo sapiens,Human,NP_000009.1,655,2.816241
...,...,...,...,...,...
648731,Imleria badia,Bay bolete (mushroom),KAF8560453.1,494,2.693727
648732,Imleria badia,Bay bolete (mushroom),KAF8560454.1,737,2.867467
648733,Imleria badia,Bay bolete (mushroom),KAF8560455.1,554,2.743510
648734,Imleria badia,Bay bolete (mushroom),KAF8560456.1,813,2.910091


In [6]:
human_protein_lengths = protein_lengths.loc[protein_lengths['Common name'] == 'Human'].copy()
# Uwaga: bez .copy() niektóre wersje Pandas mogą zwrócić widok (View).
# Może to przeszkadzać w dodaniu nowej kolumny do human_protein_lengths.
print(human_protein_lengths.head())
print()
print(human_protein_lengths.describe())

  Scientific name Common name   Protein ID  Protein length  LogLength
0    Homo sapiens       Human  NP_000005.3            1474   3.168497
1    Homo sapiens       Human  NP_000006.2             290   2.462398
2    Homo sapiens       Human  NP_000007.1             421   2.624282
3    Homo sapiens       Human  NP_000008.1             412   2.614897
4    Homo sapiens       Human  NP_000009.1             655   2.816241

       Protein length      LogLength
count   136193.000000  136193.000000
mean       692.655775       2.711540
std        746.993628       0.329892
min         12.000000       1.079181
25%        316.000000       2.499687
50%        514.000000       2.710963
75%        842.000000       2.925312
max      35991.000000       4.556194


In [7]:
citizen_incomes = pd.read_csv('3. citizen incomes.tsv', sep='\t')
citizen_incomes

,Country,Income
0,A,43806.926920
1,A,62825.327197
2,A,46660.685387
3,A,54625.879014
4,A,41976.428035
...,...,...
1995,B,55338.284370
1996,B,95699.796316
1997,B,34417.930506
1998,B,20637.868432


## Testowanie wartości średniej

Aby zilustrować podstawowe pojęcia testowania hipotez statystycznych, zaczniemy od prostych testów hipotezy, że prawdziwa wartość średnia jest równa pewnej wartości $\mu_0$, z hipotezą alternatywną, że jest od niej różna:

$$H_0: \mu = \mu_0$$
$$H_1: \mu \neq \mu_0$$
Zwróć uwagę na brak daszków nad symbolami - hipotezy dotyczą parametrów, a nie estymatorów czy jakichkolwiek innych wartości losowych.

**Zadanie 1.** Rozważ dane o log-długościach ludzkich białek w ramce danych `human_protein_lengths`. Rozważ następujące dwie hipotezy zerowe: $H_0^{(1)}: \mu = 2.711540$ oraz $H_0^{(2)}: \mu = 6$. Do weryfikacji hipotez na podstawie próby losowej użyjemy testu t-Studenta.

1. Wylosuj próbę log-długości białek o rozmiarze $N=20$.
2. Oblicz statystyki testowe jednopróbkowych testów t-Studenta przy założeniu, że odchylenie standardowe jest nieznane i jest estymowane z próby. Zwróć uwagę, jakiego estymatora wariancji należy użyć (obciążonego czy nieobciążonego). Ile statystyk testowych trzeba obliczyć, aby przetestować obie hipotezy, $H_0^{(1)}$ i $H_0^{(2)}$?
3. Użyj `tstud.ppf` (tj. funkcji kwantylowej rozkładu t-Studenta), aby wyznaczyć zbiór krytyczny na poziomie istotności 5% (tj. przy ryzyku błędu I rodzaju 5%). Zwróć uwagę na kształt zbioru krytycznego - dla naszej hipotezy alternatywnej $H_1$ jest on sumą dwóch półprostych. Ile wartości kwantyli trzeba obliczyć, aby przetestować obie hipotezy, $H_0^{(1)}$ i $H_0^{(2)}$?
  3.1. Czy na podstawie wartości statystyki testowej i zbioru krytycznego odrzucamy nasze hipotezy zerowe? Czy poprawnie wykryliśmy, która hipoteza jest prawdziwa, a która fałszywa?
4. Użyj `tstud.cdf`, aby obliczyć p-wartości. Ponownie zwróć uwagę na kształt zbioru krytycznego. Ile wartości dystrybuanty trzeba obliczyć, aby przetestować obie hipotezy, $H_0^{(1)}$ i $H_0^{(2)}$?
  4.1. Czy na podstawie p-wartości odrzucamy którąś z naszych hipotez na poziomie istotności 5%? Czy poprawnie wykryliśmy, która hipoteza jest prawdziwa, a która fałszywa?
5. Porównaj swoje wyniki z implementacją testu t-Studenta w `scipy`. Odpowiedni test został już zaimportowany w sekcji *Dane i import bibliotek*.
6. Czy któreś z założeń testu jest naruszone? Jeśli tak, to w jakim stopniu i jaki może to mieć wpływ na wynik testu?
7. Czy na podstawie wyników tego zadania można stwierdzić, że $N=20$ wystarcza, aby udowodnić, że $\mu=2.711540$? Czy odpowiedź na to pytanie zależy od $N$?

In [ ]:
# tutaj wpisz swój kod


Estimated mean: 2.7052082129437274
Value of the test statistic for H^1_0: -0.1318790992744104
Value of the test statistic for H^2_0: -68.62425557146037
We reject the hypothesis if T < -2.093024054408263 or T > 2.093024054408263
P-value for H^1_0: 0.8964664507968763
P-value for H^2_0: 3.134108172863983e-24
P-value corresponding to the critical region: 0.05000000000000464
Scipy results, H^1_0: TtestResult(statistic=-0.1318790992744104, pvalue=0.8964664507968763, df=19)
Scipy results, H^2_0: TtestResult(statistic=-68.62425557146035, pvalue=3.134108172863983e-24, df=19)


**Zadanie 2.** W tym zadaniu zobaczymy bardziej użyteczne zastosowanie testowania hipotez statystycznych: porównywanie dwóch populacji. Powiedzmy, że chcemy na podstawie próby losowej sprawdzić, czy log-długości ludzkich białek są średnio większe niż białek innego organizmu - np. podgrzybka brunatnego (rodzaju grzyba).

1. Jakie są odpowiednie hipotezy zerowa i alternatywna, jeśli chcemy na podstawie próby losowej pokazać, że ludzkie białka są dłuższe pod względem średniej log-długości?
2. Wylosuj z ramki danych `protein_lengths` próbę ludzkich białek ($N_\text{human}=20$) oraz próbę białek podgrzybka brunatnego ($N_\text{bolete}=20$).
3. Użyj dwupróbkowego testu t-Studenta zaimplementowanego w funkcji `ttest_ind` z biblioteki `scipy`, aby obliczyć p-wartość. Zwróć uwagę na argument `alternative`, a także na argumenty, które mogą odpowiadać założeniom, takim jak równość wariancji.
4. Czy na podstawie p-wartości odrzucamy $H_0$ na poziomie istotności 5%? Czy potwierdziliśmy, że ludzie mają dłuższe lub krótsze białka niż grzyb (pod względem log-długości)?
5.\*\* Zaimplementuj własny test i porównaj p-wartość. Możesz skorzystać z wzorów opisanych [tutaj](https://en.wikipedia.org/wiki/Welch%27s_t-test).
6. Co się stanie, jeśli przyjmiemy odwrotną hipotezę - że ludzie mają mniejsze log-długości białek niż grzyb? Czy teraz nasze wyniki coś potwierdziły?
  6.1. Czy wynik testu jest prawdziwy? Porównaj prawdziwe średnie log-długości obu organizmów.
  6.2. Czy wynik testu z punktu 5 jest prawdziwy?
7.\* Wykonaj test dla hipotezy zerowej, że średnie log-długości białek są równe, i hipotezy alternatywnej, że są różne (w dowolnym kierunku; tzw. *dwustronna* hipoteza alternatywna). Wyjaśnij różnicę w wynikach w porównaniu z poprzednimi punktami.
  7.1. Z grubsza rzecz biorąc, czym różni się kształt obszaru krytycznego dla jednostronnej i dwustronnej hipotezy alternatywnej?
  7.2.\* Oblicz stosunek p-wartości dla dwustronnej hipotezy alternatywnej do p-wartości dla jednostronnej hipotezy alternatywnej, że ludzkie log-długości są mniejsze niż log-długości podgrzybka. Wyjaśnij wynik.
8. Czy możemy użyć testu t-Studenta do testowania długości białek zamiast log-długości?


In [9]:
# tutaj wpisz swój kod


Scipy results for Human > Bolete alternative: TtestResult(statistic=1.207553875066569, pvalue=0.11737472456064323, df=37.65404146086758)
Scipy results for Human < Bolete alternative: TtestResult(statistic=1.207553875066569, pvalue=0.8826252754393568, df=37.65404146086758)
Scipy results for a two-sided alternative: TtestResult(statistic=1.207553875066569, pvalue=0.23474944912128645, df=37.65404146086758)
Two-sided p-value divided by the one-sided p-value: 2.0
Manually calculated value of the T statistic: 1.207553875066569
Manually calculated p-value (one-sided): 0.11737472456064313
                       Protein length  LogLength
Common name                                     
Apple                      464.063053   2.573145
Bay bolete (mushroom)      378.286030   2.443671
Chicken                    733.153531   2.732487
Chimpanzee                 742.965914   2.727545
European goshawk           740.431115   2.740182
Gorilla                    691.720583   2.707579
Human               

## Testy nieparametryczne

Dwupróbkowy test t-Studenta zakłada normalność rozkładu populacji. Gdy to założenie jest tylko nieznacznie naruszone, jak w przypadku danych o log-długościach białek, wyniki mogą nadal być wiarygodne, zwłaszcza dla dużych prób. Dla wielu zbiorów danych wraz ze wzrostem rozmiaru próby rozkład estymatora średniej zbiega do rozkładu normalnego (gwarantuje to Centralne Twierdzenie Graniczne). Oznacza to, że estymator średniej ma rozkład „bardziej normalny” niż oryginalne dane (gdzie „bardziej normalny” odnosi się do odległości między dystrybuantami). Zwiększa to odporność testu t na niewielkie odstępstwa od normalności. Jeśli jednak to założenie jest silnie naruszone, jak w przypadku nieprzekształconych danych o długościach białek, wyniki przestają być wiarygodne. Jednym ze sposobów rozwiązania tego problemu jest użycie testów nieparametrycznych. Test nieparametryczny to test, który nie opiera się na założeniu o rozkładzie danych.

Jednym z najczęściej stosowanych testów nieparametrycznych jest test U Manna-Whitneya, znany także jako dwupróbkowy test Wilcoxona. Często używa się go zamiast testu t-Studenta, gdy dane nie mają rozkładu normalnego. Hipotezy zerowe tych dwóch testów są jednak różne i ważne jest zrozumienie tej różnicy, aby uniknąć mylących wyników.

W przeciwieństwie do testu t-Studenta, test Manna-Whitneya nie testuje równości parametrów takich jak średnia - stąd nazwa *nieparametryczny*. Zamiast tego jego hipotezą zerową jest $\mathbb{P}(X > Y) = 1/2$, tj. że jeśli weźmiemy losową obserwację $X$ z pierwszej próby i losową obserwację $Y$ z drugiej próby, to równie prawdopodobne jest, że pierwsza będzie większa, jak i że będzie mniejsza od drugiej. Jednostronną hipotezą alternatywną może być np. $\mathbb{P}(X > Y) > 1/2$, tj. że obserwacje z pierwszej populacji mają tendencję do bycia większymi niż obserwacje z drugiej. W takim przypadku mówimy, że pierwsza próba jest *stochastycznie większa* od drugiej.

Uwaga na marginesie: faktyczna hipoteza zerowa testu Manna-Whitneya jest nieco inna, ale opisana powyżej jest bardzo dobrym przybliżeniem, znacznie prostszym w interpretacji i użyciu w praktyce.




**Zadanie 3.\*** Zaimplementuj własną wersję testu Manna-Whitneya. Potrzebne wzory znajdziesz [tutaj](https://en.wikipedia.org/wiki/Mann%E2%80%93Whitney_U_test) (użyj przybliżenia normalnego dla statystyki testowej).

1. Użyj swojej implementacji, aby sprawdzić, czy długości białek są większe u człowieka niż u podgrzybka brunatnego (użyj próby losowej o wybranym przez siebie rozmiarze $N$).
2. Porównaj swoje wyniki z funkcją `mannwhitneyu` z `scipy`. Zwróć uwagę na domyślne parametry, aby uzyskać identyczne wyniki.
3. Porównaj wyniki testu Manna-Whitneya z wynikami testu t-Studenta. Czy wyniki obu testów są spójne? Czy można stwierdzić, że jeden z organizmów ma dłuższe białka? Czy jest to poprawny wynik?


In [ ]:
# tutaj wpisz swój kod


Scipy test results for lengths:
TtestResult(statistic=2.8755078122880064, pvalue=0.00333359425357163, df=36.826621370440954)
MannwhitneyuResult(statistic=323.0, pvalue=0.0004604566971440039)

Manual test results for lengths:
Manually computed value of the U test statistic: 323.0
Expected value of U: 200.0 with standard deviation: 36.968455021364726
Z-score: 3.327160951922825
Manually computed p-value: 0.000438678411438298
MannwhitneyuResult(statistic=323.0, pvalue=0.00043867841143833047)


**Zadanie 4.** Ramka danych `citizen_income`, wczytana w sekcji *Dane i import bibliotek*, zawiera informacje o rocznych dochodach w USD losowo wybranych osób z dwóch krajów, oznaczonych jako Kraj `A` i Kraj `B`. Użyj odpowiedniego testu statystycznego, aby sprawdzić, czy obywatele jednego z krajów zarabiają więcej niż obywatele drugiego. Jeśli użyjesz więcej niż jednego testu i otrzymasz sprzeczne wyniki, wyjaśnij, dlaczego tak się dzieje.   

In [ ]:
# tutaj wpisz swój kod


T-test results: TtestResult(statistic=-2.081134180586558, pvalue=0.01883558596169213, df=1017.7689156871639)
U-test results: MannwhitneyuResult(statistic=670587.0, pvalue=3.827400955390909e-40)
Average incomes:                Income
Country              
A        54555.100828
B        60575.000000
Median incomes:                Income
Country              
A        54654.283542
B        30336.007047


**Zadanie 5.\*\*** W tym zadaniu zobaczymy, jak naruszenie założeń testu wpływa na rozkład statystyk testowych przy prawdziwości hipotezy zerowej.

1. Sformułuj hipotezę statystyczną dotyczącą dochodów obywateli kraju `B`, którą można przetestować dowolną wersją testu t-Studenta i w której hipoteza zerowa $H_0$ jest *prawdziwa*. Zauważ, że w tym zadaniu dopuszczamy hipotezy naruszające założenia testu.
  1.1.\* Opcjonalnie sformułuj hipotezę, którą można przetestować zarówno testem t-Studenta, jak i testem U Manna-Whitneya.
2. Które założenia Twojego testu są naruszone w tym zbiorze danych? Czy są one w przybliżeniu spełnione dla dużych rozmiarów prób?
3. Powtórz $R=5000$ razy:
  3.1. Wylosuj dwie próby, każdą o rozmiarze $N=10$.
  3.2. Oblicz wartości statystyki T Studenta, ręcznie lub za pomocą funkcji z `scipy`.
  3.3.\* Oblicz statystykę U Manna-Whitneya za pomocą funkcji z `scipy`.
  3.4. Zapisz wartości statystyk w listach.
4. Narysuj histogram przedstawiający rozkład statystyki. Czy rozkład jest poprawny (tj. czy zgadza się z teoretycznymi założeniami testów)? Jeśli nie, jak wpływa to na wyniki testu?
  4.1.\* Narysuj na histogramach gęstość teoretycznego rozkładu statystyk testowych (przy prawdziwości hipotezy zerowej).
  4.2. Oblicz prawdopodobieństwo, że Twój test popełni na tym zbiorze danych błąd fałszywie dodatni (tj. że niepoprawnie odrzuci prawdziwą hipotezę zerową; tj. że statystyka testowa znajdzie się w teoretycznie wyznaczonym obszarze krytycznym).
5. Czy przeanalizowaliśmy wszystkie możliwe sposoby, w jakie naruszone założenia mogą wpływać na wyniki testu? Jeśli nie, jakie inne problemy lub błędy mogą być spowodowane naruszonymi założeniami?
6. Co się stanie, jeśli zamiast dochodów obywateli użyjesz log-długości białek?




In [ ]:
# tutaj wpisz swój kod


Probability of Type I error for Mann-Whitney: 0.059
Probability of Type I error for Student: 0.0392


## Testowanie rozkładu danych

Do tej pory omówiliśmy testowanie wartości parametrów oraz porównywanie dwóch populacji. Kolejną ważną klasą są testy *zgodności* (ang. *goodness-of-fit*), służące do sprawdzania, czy można założyć, że próba pochodzi z pewnego teoretycznego rozkładu, lub do sprawdzania, czy dwie zmienne mają takie same rozkłady. Prawdopodobnie najczęściej używane testy to:
* test chi-kwadrat, przeznaczony dla danych kategorycznych
* test Kołmogorowa-Smirnowa, przeznaczony dla danych ciągłych
* test Shapiro-Wilka, przeznaczony specjalnie dla rozkładu normalnego

**Zadanie 6.** W tym zadaniu użyjemy testu Shapiro-Wilka, aby spróbować wykryć, czy niektóre rozkłady z poprzednich zadań nie są normalne.

1. Użyj testu Shapiro-Wilka (`scipy.shapiro`), aby sprawdzić, czy dochody obywateli z Zadania 4 mają rozkład normalny (osobno dla każdego kraju). Czy możemy stwierdzić, że dochody w kraju B mają rozkład normalny? A w kraju A?
2.\*\* Użyj testu Shapiro-Wilka, aby sprawdzić rozkład statystyk testowych z Zadania 5. Czy możemy stwierdzić, że mają one rozkład normalny? Czy odpowiedź zależy od wartości parametrów $R$ (liczby prób) i $N$ (rozmiaru próby)? Jeśli tak, to w jaki sposób?




In [ ]:
# tutaj wpisz swój kod


Testing the distribution of citizen incomes:
Country A: ShapiroResult(statistic=0.9987704777213762, pvalue=0.7355847922419261)
Country B: ShapiroResult(statistic=0.5760313435167443, pvalue=7.3099920695338e-44)

Testing the distribution of statistics from Exercise 6:
Test result for Student's T statistic: ShapiroResult(statistic=0.9948056325905207, pvalue=1.9825988394300593e-12)
Test result for Mann-Whitney's U statistic: ShapiroResult(statistic=0.9991477739009211, pvalue=0.014051880866807481)


**Zadanie 7**. Piekarnia chce wprowadzić do produkcji nowy rodzaj chleba. Opracowała trzy potencjalne receptury i poprosiła próbę 120 osób o wybranie ulubionej. Recepturę 1 wybrało 30 osób, recepturę 2 - 50, a recepturę 3 - 40 osób. Czy oznacza to, że receptury będą się różnić popularnością? Sprawdź to za pomocą testu zgodności chi-kwadrat (`scipy.chisquare`).  

In [ ]:
# tutaj wpisz swój kod


Power_divergenceResult(statistic=5.0, pvalue=0.0820849986238988)

**Zadanie 8.\*\*** W tym zadaniu użyjemy testu zgodności, aby złamać następującą wiadomość zaszyfrowaną [szyfrem Cezara](https://pl.wikipedia.org/wiki/Szyfr_Cezara):

*AOLJHLZHYJPWOLYPZVULVMAOLLHYSPLZARUVDUHUKZPTWSLZAJPWOLY
ZPAPZHAFWLVMZBIZAPABAPVUJPWOLYPUDOPJOLHJOSLAALYPUAOLWSH
PUALEAPZZOPMALKHJLYAHPUUBTILYVMWSHJLZKVDUAOLHSWOHILA*

Niech $X_i$ oznacza częstość $i$-tej litery alfabetu w powyższej wiadomości. Użyj odpowiedniego testu zgodności, aby znaleźć takie przesunięcie alfabetu, dla którego rozkład $X_i$ jest statystycznie nieodróżnialny od rozkładu liter w języku angielskim. Częstość każdej litery w języku angielskim oraz funkcja przesuwająca alfabet o zadaną liczbę liter znajdują się w komórce poniżej.





In [ ]:
letter_frequency = {
    'E': 0.1202,
    'T': 0.091,
    'A': 0.0812,
    'O': 0.0768,
    'I': 0.0731,
    'N': 0.0695,
    'S': 0.0628,
    'R': 0.0602,
    'H': 0.0592,
    'D': 0.0432,
    'L': 0.0398,
    'U': 0.0288,
    'C': 0.0271,
    'M': 0.0261,
    'F': 0.023,
    'Y': 0.0211,
    'W': 0.0209,
    'G': 0.0203,
    'P': 0.0182,
    'B': 0.0149,
    'V': 0.0111,
    'K': 0.0069,
    'X': 0.0017,
    'Q': 0.0011,
    'J': 0.0011,
    'Z': 0.0007
}

def shift_string(s, n):
  """
  Zwraca napis otrzymany z napisu wejściowego s
  przez przesunięcie każdej litery o n miejsc w alfabecie
  """
  return ''.join(map(chr, (65 + (o-65+n)%26 for o in map(ord, s))))

In [ ]:
# tutaj wpisz swój kod


Chi-square test p-values for different shifts: 
0 0.0
1 0.0
2 0.0
3 0.0
4 0.0
5 0.0
6 0.0
7 0.0
8 0.0
9 0.0
10 0.0
11 0.0
12 0.0
13 0.0
14 0.0
15 0.0
16 0.0
17 0.0
18 0.0
19 0.1543
20 0.0
21 0.0
22 0.0
23 0.0
24 0.0
25 0.0
Decoded message:
THECAESARCIPHERISONEOFTHEEARLIESTKNOWNANDSIMPLESTCIPHERSITISATYPEOFSUBSTITUTIONCIPHERINWHICHEACHLETTERINTHEPLAINTEXTISSHIFTEDACERTAINNUMBEROFPLACESDOWNTHEALPHABET


## Testowanie wielu hipotez




Wykonując testy statystyczne, odrzucamy hipotezę zerową, gdy p-wartość jest poniżej ustalonego progu, tradycyjnie 0.05. Konsekwencją tego podejścia jest to, że 5% wyników pozytywnych będzie fałszywie pozytywnych (błędy I rodzaju; niepoprawne odrzucenia prawdziwej hipotezy zerowej), co może oznaczać ogromną liczbę, gdy w badaniach na dużą skalę wykonuje się tysiące testów.

Aby ograniczyć liczbę wyników fałszywie pozytywnych, stosujemy *poprawki na wielokrotne testowanie*. Jedną z najbardziej użytecznych jest poprawka Benjaminiego-Hochberga, która kontroluje odsetek fałszywych odkryć (ang. *False Discovery Rate*, FDR), tj. odsetek wyników fałszywie pozytywnych wśród wszystkich wyników pozytywnych. Zwróć uwagę na różnicę w stosunku do poziomu istotności: odsetek fałszywych odkryć to odsetek wyników fałszywie pozytywnych wśród wszystkich pozytywnych *wyników* testów, podczas gdy poziom istotności to odsetek wyników fałszywie pozytywnych wśród *prawdziwie negatywnych* przypadków (gdy $H_0$ jest prawdziwa).

Założenia i szczegóły poprawki Benjaminiego-Hochberga (oraz innych popularnych poprawek) były omawiane na wykładzie; warto je teraz powtórzyć. Jednym z najważniejszych założeń jest to, że p-wartości pochodzą ze zbioru niezależnych testów.

Popularnym sposobem przeprowadzenia poprawki Benjaminiego-Hochberga jest przekształcenie p-wartości $p_1, \dots, p_m$ tak, aby otrzymać tzw. *q-wartości* $q_1, \dots, q_m$, takie że uzyskujemy FDR na poziomie $Q$, gdy przyjmujemy wszystkie hipotezy $H_i$, dla których $q_i \leq Q$.


Nie myl odsetka fałszywych odkryć (FDR) z innymi podobnie nazwanymi miarami, takimi jak odsetek wyników fałszywie pozytywnych (ang. *False Positive Rate*, FPR)! FDR to stosunek liczby wyników fałszywie pozytywnych do liczby wszystkich zgłoszonych wyników pozytywnych, natomiast FPR to stosunek liczby wyników fałszywie pozytywnych do liczby prawdziwych hipotez zerowych. Listę innych popularnych miar znajdziesz [tutaj](https://en.wikipedia.org/wiki/Confusion_matrix).   

W następnym zadaniu użyjemy *symulacji stochastycznych* do wygenerowania własnych danych. Najpierw jednak porozmawiajmy o tym, co oznacza symulowanie zmiennej losowej.

*Czas na teorię.* Termin *symulowanie zmiennej losowej* jest mylący dla wielu studentów, zwłaszcza tych, którzy uważali na zajęciach z rachunku prawdopodobieństwa. Przypomnijmy, że zmienna losowa $X$ jest funkcją z przestrzeni zdarzeń elementarnych $\Omega$ w zbiór liczb rzeczywistych (lub całkowitych). Gdy niektórzy studenci słyszą „zasymulujmy ciąg zmiennych losowych $X_1, \dots, X_n$”, zastanawiają się: jak symulować ciąg funkcji? Otóż wcale nie o to chodzi.

Definiujemy zmienne losowe jako funkcje, ponieważ chcemy mieć matematyczny model losowości. Jest to wygodny sposób na ścisłe zdefiniowanie, co znaczy, że zmienna $X$ przyjmuje pewną wartość z pewnym prawdopodobieństwem. Możemy go na przykład użyć, aby formalnie zdefiniować $\mathbb{P}(X=1) = \mathbb{P}(\{ \omega: X(\omega)  =1 \})$. Potrzebujemy tej definicji, ponieważ pojęcie prawdopodobieństwa jest zdefiniowane na przestrzeni $\Omega$, a nie na liczbach rzeczywistych. Nie w ten sposób jednak interpretujemy i wykorzystujemy zmienne losowe w praktyce.

Przypomnijmy, że liczba oczek na zestawie kostek do gry jest ciągiem zmiennych losowych $X_1, \dots, X_n$, zanim rzucisz kostkami. Gdy nimi rzucisz, otrzymujesz realizacje zmiennych losowych, $X_1(\omega), \dots, X_n(\omega)$, które są ciągiem liczb (często nazywanych *liczbami losowymi*). Zmierzenie wyniku doświadczenia losowego odpowiada ustaleniu $\omega$ w ciągu zmiennych losowych. Dlatego możemy interpretować zmienne losowe jako zmienne liczbowe. Definiowanie ich jako funkcji jest konieczne tylko dlatego, że w przeciwnym razie wyrażenia takie jak $\mathbb{P}(X=1)$ nie miałyby żadnego matematycznego sensu.

„Ustalenie $\omega$” to również to, co rozumiemy przez symulowanie ciągu zmiennych losowych: chcemy otrzymać ciąg liczb $x_1, \dots, x_n$, które są realizacjami ciągu zmiennych losowych $X_1, \dots X_n$, tak że $x_i = X_i(\omega)$. Symulowanie nie oznacza jednak, że wybieramy $\omega$ ręcznie. Pamiętaj, że $\omega$ to przede wszystkim model matematyczny, a nie byt fizyczny. Zamiast wybierać je ręcznie, polegamy na generatorach liczb losowych, które „ustalają $\omega$” za nas.

Zetknąłeś się już z kilkoma generatorami liczb losowych. Zestaw kostek do gry jest jednym z nich. Gdy rzucasz kostkami, możesz wyobrazić sobie, że czynność rzutu odpowiada ustaleniu $\omega$ w ciągu zmiennych losowych - i w rezultacie otrzymujesz ciąg liczb. Prawdopodobnie używałeś też w swoich programach generatorów liczb pseudolosowych. Są to algorytmy zwracające liczby pseudolosowe z przedziału od 0 do 1. Gdy uruchamiasz taki algorytm, możesz wyobrazić sobie, że odpowiada to ustaleniu $\omega$ w zmiennej losowej $X\sim U([0, 1])$ - i w rezultacie otrzymujesz liczbę losową.

Te dwa generatory liczb losowych (kostki i algorytm) zwracają liczby o bardzo prostych rozkładach. Możemy jednak symulować liczby losowe o wielu innych rozkładach, takich jak rozkład normalny. Istnieje wiele technik, które to umożliwiają; zazwyczaj polegają one na przekształcaniu liczb losowych o rozkładzie jednostajnym na przedziale od 0 do 1 (formalnie: liczb będących realizacją ciągu zmiennych losowych $X_i$ takich, że $X_i \sim_{iid} U([0, 1])$) (iid oznacza niezależne o jednakowym rozkładzie, czyli próbę prostą). Na tym kursie nie musisz przejmować się tymi technikami: będziemy symulować zmienne losowe za pomocą algorytmów już zaimplementowanych w bibliotekach Pythona.




**Zadanie 9.** W tym zadaniu nauczymy się, jak używać poprawki Benjaminiego-Hochberga do kontrolowania odsetka fałszywych odkryć (FDR), a także jak symulować własne zbiory danych. Wygenerujemy zbiór prób z dwóch różnych rozkładów i przetestujemy wartości ich średnich, otrzymując zbiór testów, w których część hipotez zerowych jest prawdziwa, a część fałszywa.

1. Wygeneruj $M=1000$ prób losowych o rozmiarze $N=10$ ze standardowego rozkładu normalnego oraz $M=1000$ prób o rozmiarze $N=10$ z rozkładu normalnego o wartości oczekiwanej $\mu=1$. Użyj funkcji `norm.rvs` z `scipy` (w razie potrzeby poszukaj jej dokumentacji w internecie).
2. Dla każdej próby wykonaj jednopróbkowy test t-Studenta, aby sprawdzić, czy średnia jest niezerowa. Zapisz p-wartości.
  2.1. Ile hipotez zerowych jest prawdziwych, a ile fałszywych?
3. Odrzuć hipotezy zerowe na poziomie istotności 5% (tj. gdy p-wartości są mniejsze niż 0.05).
  3.1. Ile wyników pozytywnych (tj. odrzuceń hipotezy zerowej) otrzymałeś?
  3.2. Ile otrzymałeś wyników prawdziwie pozytywnych?
  3.3. Ile otrzymałeś wyników fałszywie pozytywnych?
  3.4. Ilu wyników fałszywie pozytywnych spodziewałeś się przy danym poziomie istotności?
  3.5. Oblicz uzyskany odsetek wyników fałszywie pozytywnych (FPR). Czy jest bliski poziomowi istotności? Dlaczego tak/nie?
  3.6. Oblicz uzyskany odsetek fałszywych odkryć (FDR).
  3.7. Które z poprzednich punktów zależą od $N$? Które zależą od $M$? Podaj argument teoretyczny.
4. Przeprowadź poprawkę Benjaminiego-Hochberga na swoich p-wartościach za pomocą funkcji `fdrcorrection` z biblioteki `statsmodels`. Odrzuć hipotezę zerową na poziomie FDR 10%.
  4.1. Ile wyników pozytywnych otrzymałeś?
  4.2. Ile otrzymałeś wyników prawdziwie pozytywnych?
  4.3. Ile otrzymałeś wyników fałszywie pozytywnych?
  4.4. Oblicz uzyskany odsetek wyników fałszywie pozytywnych (FPR). Czy jest bliski poziomowi istotności? Dlaczego tak/nie?
  4.5. Oblicz uzyskany odsetek fałszywych odkryć (FDR). Czy jest bliski założonemu FDR 10%? Dlaczego tak/nie?
  4.6. Ilu wyników fałszywie pozytywnych spodziewałeś się przy danym poziomie FDR?
5. Załóżmy, że wszystkie próby zostały wylosowane ze standardowego rozkładu normalnego. Jaki byłby efekt zastosowania poprawki Benjaminiego-Hochberga? Ilu wyników fałszywie pozytywnych spodziewałbyś się otrzymać?
6. \* Na podstawie opisu procedury Benjaminiego-Hochberga z [tego artykułu na Wikipedii](https://en.wikipedia.org/wiki/False_discovery_rate#Benjamini%E2%80%93Hochberg_procedure) ustal, jak obliczyć q-wartości na podstawie listy p-wartości. Napisz funkcję, która przyjmuje wektor p-wartości i zwraca odpowiadające im q-wartości. Porównaj swoje wyniki z funkcją `fdrcorrection` z biblioteki `statsmodels` na wektorze p-wartości (0.01, 0.1, 0.01, 0.2, 0.01, 0.1).

In [ ]:
# tutaj wpisz swój kod


My q-values: [0.02 0.12 0.02 0.2  0.02 0.12]
Statsmodels: [0.02 0.12 0.02 0.2  0.02 0.12]

Testing
Number of tests done: 1000
Number of positives: 431
Number of true positives: 409
Number of false positives: 22
"Raw" FPR: 0.044
"Raw" FDR: 0.05104408352668213
HB number of positives: 409
HB number of true positives: 392
HB number of false positives: 17
HB FPR: 0.034
HB FDR: 0.04156479217603912


**Zadanie 10.\*** W poprzednim zadaniu mogłeś zauważyć, że wyniki różnią się między uruchomieniami. Każde losowanie $2M$ prób daje inny FDR. W tym zadaniu przeanalizujemy rozkład FDR.
1. Powtórz obliczenie „surowego” FDR (tj. przed poprawką) oraz FDR „BH” (po poprawce) $R=1000$ razy. Oblicz także odsetek wyników prawdziwie pozytywnych (ang. *True Positive Rate*, TPR), tj. frakcję poprawnie zidentyfikowanych przypadków pozytywnych.
2. Narysuj histogramy porównujące FDR i TPR przed i po poprawce BH.
  2.1. Czy stosowanie poprawki Benjaminiego-Hochberga ma jakieś negatywne skutki?
3. Oblicz średni FDR po poprawce. Czy jest bliski założonemu poziomowi 10%? Dlaczego tak/nie?       

In [ ]:
# tutaj wpisz swój kod


**Zadanie 11.** W każdym z poniższych eksperymentów badawczych albo jakaś procedura została zastosowana niepoprawnie, albo jest poprawna, ale można ją ulepszyć. Dla każdego punktu określ, czy wszystkie założenia są spełnione oraz czy metodologię można ulepszyć (np. stosując inny test statystyczny lub inną procedurę poprawki). Jeśli założenia są naruszone, omów możliwy skutek tego rodzaju naruszenia. Za dodatkowe punkty poprzyj swoje twierdzenia argumentami teoretycznymi lub symulacjami.
1. Badacz bada wpływ spożycia glukozy na samopoczucie szczurów. Badacz podzielił $N=20$ szczurów na dwie grupy: eksperymentalną z nadmiarem glukozy w diecie oraz kontrolną z normalną dietą. Po tygodniu badacz zmierzył masę ciała szczurów, długość snu, poziom glukozy we krwi, obwód brzucha, procentową zawartość tkanki tłuszczowej oraz poziom lęku (za pomocą testu podwyższonego labiryntu krzyżowego). Badacz porównał każdy czynnik między grupą eksperymentalną a kontrolną za pomocą dwupróbkowego testu t-Studenta dla obserwacji niesparowanych, wcześniej sprawdziwszy testem Shapiro-Wilka, że porównywane zmienne mają rozkład normalny. Następnie badacz skorygował p-wartości z testów t-Studenta na wielokrotne testowanie za pomocą poprawki Benjaminiego-Hochberga na poziomie FDR 20%.
2. Badacz bada wpływ spożycia niewielkich ilości octu na wzrost poziomu glukozy we krwi (BGC) u myszy. Badacz podzielił $N=20$ myszy na równoliczne grupy eksperymentalną i kontrolną. Na początku eksperymentu zmierzył BGC wszystkich myszy. Następnie przez cztery dni myszy z obu grup otrzymywały dietę wzbogaconą w glukozę. Myszy z grupy eksperymentalnej otrzymywały dodatkowo w diecie niewielkie ilości octu. Po czterech dniach ponownie zmierzono BGC wszystkich myszy. Następnie w każdej grupie badacz przetestował wzrost BGC za pomocą dwupróbkowego testu t-Studenta dla obserwacji niesparowanych.
3. Badacz bada wpływ sztucznych słodzików na poziom glukozy we krwi (BGC) u myszy. Badacz podzielił $N=100$ myszy na dziesięć równolicznych grup: dziewięć eksperymentalnych i jedną kontrolną. Myszy w grupie kontrolnej otrzymywały normalną dietę. W poszczególnych grupach eksperymentalnych myszy otrzymywały pokarm wzbogacony odpowiednio w: glukozę; sukralozę; aspartam; erytrytol; stewię; ksylitol; sacharynę; acesulfam K; mannitol. Po czterech dniach badacz zmierzył BGC wszystkich myszy. Następnie badacz użył dwupróbkowego testu t-Studenta dla obserwacji niesparowanych, aby porównać średnie BGC między dziesięcioma grupami. W celu uwzględnienia wielokrotnego testowania badacz zastosował procedurę Benjaminiego-Hochberga na poziomie FDR 20%.
4. Badacz bada wpływ spożycia soli na obwód brzucha (AC) u myszy. Badacz podzielił $N=20$ myszy na równoliczne grupy eksperymentalną i kontrolną. Na początku eksperymentu zmierzono AC wszystkich myszy. Myszy z grupy eksperymentalnej otrzymywały dietę wzbogaconą w sól. Po tygodniu ponownie zmierzono AC, a badacz w każdej grupie zastosował dwupróbkowy test t-Studenta dla obserwacji sparowanych. Wyniki testu w każdej grupie były nieistotne i badacz stwierdził, że spożycie soli nie zwiększa obwodu brzucha w ciągu tygodnia.     

*Podziękowania.* Zadanie 8 zostało zainspirowane dwoma następującymi źródłami: http://practicalcryptography.com/cryptanalysis/text-characterisation/chi-squared-statistic/ oraz https://ibmathsresources.com/2014/06/15/using-chi-squared-to-crack-codes/.
Zadanie 11 zostało częściowo zainspirowane następującym źródłem: https://stats.libretexts.org/Bookshelves/Applied_Statistics/Biological_Statistics_(McDonald)/06%3A_Multiple_Tests/6.01%3A_Multiple_Comparisons

<center><img src='https://drive.google.com/uc?export=view&id=12CrUdXDAiltLBT26sG7HZ_HciIhvGyT8'></center>|